# 00 · Simulator sanity check (Gate G1)

Before any phone data is used, this notebook establishes four things:

1. LIBERO runs here, and we can read exact object poses.
2. **Our action interface is right**: a hand-coded pick-and-place built from privileged object poses succeeds on LIBERO's 50 fixed evaluation scenes. If this fails, every later number is meaningless, so nothing downstream starts until it passes.
3. **Which parts of the motion are sensitive** (where you grasp vs. where you place). This tells us what retargeting must get right.
4. Our observations match LeRobot's `lerobot/libero` conventions (image orientation, 8-D state), so policies trained on our data and on the official demos see identical inputs.

**Runtime:** GPU (T4) recommended. A CPU runtime also works, but section 4 renders images in software (~5 steps/s).
Code to read alongside: `p2p/sim/env.py`, `p2p/sim/controller.py`, `p2p/sim/scripted.py`.

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
os.environ["MUJOCO_GL"] = "egl"   # headless OpenGL; must be set before mujoco/robosuite are imported
if IN_COLAB:
    # LIBERO via LeRobot (Linux only). Takes a few minutes; restart is NOT needed afterwards.
    !pip install -q "lerobot[libero]==0.6.1" imageio-ffmpeg

In [ ]:
# Get the project code: either clone your GitHub repo or upload phone2panda_code.zip
REPO_URL = "https://github.com/LukasWill/phone2panda.git"   # set to "" to upload phone2panda_code.zip instead
CODE_DIR = "/content/phone2panda" if IN_COLAB else os.path.abspath("..")
if IN_COLAB and not os.path.exists(os.path.join(CODE_DIR, "p2p")):
    if REPO_URL:
        !git clone -q {REPO_URL} {CODE_DIR}
    else:
        from google.colab import files
        uploaded = files.upload()                      # pick phone2panda_code.zip
        !unzip -q -o {list(uploaded)[0]} -d {CODE_DIR}
sys.path.insert(0, CODE_DIR)

import json, time
import numpy as np
import pandas as pd
from p2p.sim.env import TaskEnv, lerobot_state, MAX_STEPS
from p2p.sim.scripted import WaypointFollower, plan_bowl_to_plate, run_episode
from p2p.stats import fmt_rate, wilson_ci
SUMMARY = {}

## 1 · The environment

`render=False` builds the simulator without cameras. That is ~15× faster and enough whenever only the outcome (success or not) matters.

In [ ]:
env = TaskEnv(render=False)
raw = env.reset(init_state_id=0)
print("task id in libero_goal:", env.task_id, "| fixed eval scenes:", env.init_states.shape)
s = env.scene()
for k, v in vars(s).items():
    print(f"{k:13s}", np.round(v, 3).tolist())
print("8-D LeRobot state:", np.round(lerobot_state(raw), 3).tolist())

**How to read this**
- World frame: x points forward from the robot, y left, z up. The table top is at z ≈ 0.898.
- `eef_R` ≈ diag(1, −1, −1): the gripper points straight down and its fingers close along world y.
- Bowl and plate positions are their *bottom centres*. The plate sits ~16 cm beyond the bowl.
- State = [eef xyz, eef axis-angle, finger joints]. The axis-angle is ≈ (π, 0, 0) because "pointing down" is a 180° rotation. That is right next to the axis-angle discontinuity: a small wrist roll can flip it to ≈ (−π, 0, 0). LeRobot's data has the same quirk, so we keep it identical rather than "fix" it.

## 2 · Gate G1: hand-coded pick-and-place on the 50 evaluation scenes

The plan is a list of waypoints: pre-grasp above the bowl rim → descend → close → lift → carry → lower onto the plate → open → retreat.
`PoseController` turns "go to this pose" into LIBERO's delta action: `a_pos = clip(target − current) / 0.05`, `a_rot = log(R_target · R_currentᵀ) / 0.5`.
The fingers straddle the bowl wall, so they must close along the radial direction (see `grasp_yaw_for_radial`).

In [ ]:
results, t0 = [], time.time()
for i in range(50):
    env.reset(init_state_id=i)
    results.append(run_episode(env, WaypointFollower(plan_bowl_to_plate(env.scene()))))
n_ok = sum(r["success"] for r in results)
steps = [r["steps"] for r in results]
print("scripted success:", fmt_rate(n_ok, 50), f"| mean episode length {np.mean(steps):.0f} steps (limit {MAX_STEPS}) | {time.time() - t0:.0f}s")
print("failures (scene, last waypoint):", [(i, r["last_waypoint"]) for i, r in enumerate(results) if not r["success"]])
SUMMARY["scripted_success"] = f"{n_ok}/50"
SUMMARY["scripted_mean_steps"] = round(float(np.mean(steps)))

## 3 · What must retargeting get right? (≈3–4 min)

We perturb one thing at a time and re-run on 12 scenes:
- where on the rim we grasp (angle),
- the grasp position,
- the place position.

A human demo will be imprecise in all three. This shows which imprecision actually costs success.

In [ ]:
rng = np.random.default_rng(0)
probes = {
    "nominal":              lambda: {},
    "rim angle ±60°":       lambda: dict(rim_angle=np.pi / 2 + rng.uniform(-1.05, 1.05)),
    "grasp xy ±2 cm":       lambda: dict(grasp_offset=np.r_[rng.uniform(-0.02, 0.02, 2), 0]),
    "place xy ±2 cm":       lambda: dict(place_offset=np.r_[rng.uniform(-0.02, 0.02, 2), 0]),
    "place xy ±4 cm":       lambda: dict(place_offset=np.r_[rng.uniform(-0.04, 0.04, 2), 0]),
}
rows = []
for name, kw in probes.items():
    ok = 0
    for i in range(12):
        env.reset(init_state_id=i)
        ok += run_episode(env, WaypointFollower(plan_bowl_to_plate(env.scene(), **kw())))["success"]
    lo, hi = wilson_ci(ok, 12)
    rows.append({"perturbation": name, "success": f"{ok}/12", "95% CI": f"{100*lo:.0f}-{100*hi:.0f}%"})
sens = pd.DataFrame(rows); SUMMARY["sensitivity"] = {r["perturbation"]: r["success"] for r in rows}
sens

**What to look for.** The plate is 13.7 cm across and the bowl 11.1 cm. If the bowl lands more than ~2–3 cm off-centre, it rests on the plate's rim, tilts, and is often not counted as "on" the plate. Grasp choices around the rim are much more forgiving.

So the *place* location is the part of your demo that must transfer accurately. That is why we will express it relative to the plate (object-relative retargeting) rather than in absolute table coordinates.

## 4 · Rendered rollout (what a policy sees)

Left: agentview camera. Right: wrist camera. Both are rotated 180° exactly as LeRobot does.

In [ ]:
import imageio
from IPython.display import Video, display
renv = TaskEnv(render=True, camera_size=256)
renv.reset(init_state_id=0)
follower = WaypointFollower(plan_bowl_to_plate(renv.scene()))
frames, done, t0 = [], False, time.time()
while not done:
    fr = renv.frame(size=None)
    frames.append(np.hstack([fr["observation.images.image"], fr["observation.images.image2"]]))
    _, success, done = renv.step(follower.act(renv.scene()))
print(f"success={success} in {renv.t} steps; rendering speed {renv.t / (time.time() - t0):.1f} steps/s")
imageio.mimsave("scripted_scene0.mp4", frames, fps=20)
SUMMARY["render_steps_per_s"] = round(renv.t / (time.time() - t0), 1)
display(Video("scripted_scene0.mp4", embed=True, width=512))

## 5 · Do our observations match `lerobot/libero`?

We load one official demo of the same task and put its first frame next to ours. The scenes differ slightly (different initial states), but both of these must agree:
- **orientation**: the robot is on the same side of the image, and the table is not upside down;
- **state**: the starting gripper position is the same to within a few cm.

This downloads ~100–300 MB of the dataset. It only needs to work once; if it fails, Gate G1 does not depend on it.

In [ ]:
import matplotlib.pyplot as plt
from p2p.sim.env import TASK_LANGUAGE
try:
    from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
    meta = LeRobotDatasetMetadata("lerobot/libero")
    ep = next(int(r["episode_index"]) for r in meta.episodes if TASK_LANGUAGE in (r.get("tasks") or []))
    n_task_eps = sum(TASK_LANGUAGE in (r.get("tasks") or []) for r in meta.episodes)
    ds = LeRobotDataset("lerobot/libero", episodes=[ep])
    item = ds[0]
    theirs = (item["observation.images.image"].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    their_state = item["observation.state"].numpy()
    renv.reset(init_state_id=0)
    ours = renv.frame(size=None)
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(renv.raw_obs["agentview_image"]); ax[0].set_title("raw LIBERO render (unrotated)")
    ax[1].imshow(ours["observation.images.image"]); ax[1].set_title("ours (rotated 180°)")
    ax[2].imshow(theirs); ax[2].set_title(f"lerobot/libero episode {ep}")
    for a in ax: a.axis("off")
    plt.show()
    print("official demos of this task in lerobot/libero:", n_task_eps)
    print("our state   :", np.round(ours["observation.state"], 3).tolist())
    print("their state :", np.round(their_state, 3).tolist())
    d = np.linalg.norm(ours["observation.state"][:3] - their_state[:3])
    print(f"start gripper position differs by {100 * d:.1f} cm")
    SUMMARY["official_demos_for_task"] = n_task_eps
    SUMMARY["start_eef_diff_cm"] = round(100 * float(d), 1)
except Exception as e:
    print("Convention check skipped:", type(e).__name__, e)
    SUMMARY["convention_check"] = f"skipped: {type(e).__name__}" 

## Gate G1 summary

**Pass** when the scripted success is ≥ 40/50 (it should be ~50/50) and, in section 5, the middle and right images have the same orientation.

Copy the printed JSON back to Claude.

In [ ]:
ok = int(SUMMARY["scripted_success"].split("/")[0]) >= 40
print("GATE G1:", "PASS" if ok else "FAIL", "| also confirm the image orientation in section 5 by eye")
print(json.dumps(SUMMARY, indent=1, default=str))